In [17]:
import pandas as pd
import numpy as np
import os
import json
import subprocess


def get_gene_info_by_gene_id(workdir: str, species_name: str, gene_id: str) -> dict:
    """Returns a dictionary containing pathway name, pathway ID, NCBI ID, and protein name for a given gene ID.

    Args:
        workdir: The root directory containing the data files.
        species_name: The name of the species, used to determine the location and name of the data files.
        gene_id: The ID of the gene, used to lookup related information in the KEGG pathway database and NCBI ID database.

    Returns:
        A dictionary containing pathway name, pathway ID, NCBI ID, and protein name. If no corresponding information is found,
        the respective value will be 'N/A'.
    """
    # Format species_name and gene_id
    species_name = species_name.lower().replace(' ', '_')
    gene_id = gene_id.upper()

    # files path
    kegg_file = os.path.join(workdir, species_name, 'kegg.tsv')
    pathway_file = os.path.join(workdir, species_name, 'pathway_id.tsv')
    ncbi_id_file = os.path.join(workdir, species_name, 'ncbi_id.tsv')
    location_file = os.path.join(workdir, species_name, 'location.tsv')
    go_file = os.path.join(workdir, species_name, 'go.tsv')

    # Read data files
    df_kegg = pd.read_csv(kegg_file, sep='\t')
    df_pathway = pd.read_csv(pathway_file, sep='\t', names=['Id', 'Pathway'])
    df_ncbi_id = pd.read_csv(ncbi_id_file, sep='\t', names=['NCBI ID', 'Gene ID', 'Protein'])
    df_location = pd.read_csv(location_file, sep='\t')
    df_go = pd.read_csv(go_file, sep='\t')

    # Get pathway name and ID as a dictionary
    pathway_name_list = df_kegg.loc[df_kegg['Gene'] == gene_id, 'Pathway'].tolist()
    pathway_name = pathway_name_list if pathway_name_list else 'N/A'
    pathway_id_list = df_pathway.loc[df_pathway['Pathway'].isin(pathway_name_list), 'Id'].tolist()
    pathway_id = pathway_id_list if pathway_id_list else 'N/A'
    pathway_dict = dict(zip(pathway_id, pathway_name))

    # Get NCBI ID and protein name
    ncbi_id = df_ncbi_id.loc[df_ncbi_id['Gene ID'] == 'mtm:' + gene_id, 'NCBI ID'].tolist()
    ncbi_id = ncbi_id[0] if ncbi_id else 'N/A'
    protein = df_ncbi_id.loc[df_ncbi_id['Gene ID'] == 'mtm:' + gene_id, 'Protein'].tolist()
    protein = protein[0] if protein else 'N/A'

    # Get gene location information
    gene_location = df_location.loc[df_location['Geneid'] == gene_id, ['Chr', 'Start', 'End', 'Strand', 'Length']].to_dict(orient='records')

    # Get GO information
    go_id = df_go.loc[df_go['Gene'] == gene_id, ['GOID', 'DEFINITION', 'ONTOLOGY', 'TERM']].to_dict(orient='records')
    go_id = go_id if go_id else 'N/A'


    # Create and return the gene dictionary
    dict_gene = {
        gene_id: {
            'location': gene_location,
            'NCBI ID': ncbi_id,
            'Protein': protein,
            'Pathway': pathway_dict,
            'GO': go_id,
        }
    }

    return dict_gene


def get_gene_list_by_pathway(workdir: str, species_name: str, pathway_name: str) -> list:
    """根据菌种pathway的名字，找到对应的pathway中基因的list

    Args:
    * species_name: 菌种的名字
    * pathway_name: pathway的名字
    * workdir: 工作目录
    ----------
    Returns: 
        gene_list: Pathway对应的基因的list
    """
    # 空格换成_，大写转小写
    species_name = species_name.replace(' ', '_').lower()

    # 读取kegg_filter表
    kegg_filter_path = os.path.join(workdir, species_name, 'kegg.tsv')
    df_kegg_filter = pd.read_csv(kegg_filter_path, sep='\t')

    # 根据KEGG_filter表的表头，找到对应的基因的list
    df = df_kegg_filter[df_kegg_filter['Pathway'] == pathway_name]
    gene_list = df['Gene'].tolist()
    gene_list_info = []

    for gene in gene_list:
        # 调用get_gene_info_by_gene_id函数，获取基因的go注释
        gene_info = get_gene_info_by_gene_id(workdir, species_name, gene)
        # 将每一个基因的go注释，添加到gene_list_info中
        gene_list_info.append(gene_info)

    return gene_list_info


def get_gene_list_exp_by_pathway(workdir: str, species_name: str, pathway_name: str) -> pd.DataFrame:
    """ 根据pathway的名字，依据exp_gsm表，返回对应基因的表达谱

    Args:
    * workdir: 工作目录
    * species_name: 菌种的名字
    * pathway_name: pathway的名字
    ----------
    Returns:
        df: 以GSM ID为表头的表达谱的dataframe
    """
    # 空格换成_，大写转小写
    species_name = species_name.replace(' ', '_').lower()

    # 读取exp_gsm表
    exp_path = os.path.join(workdir, species_name, 'exp_gsm.tsv')
    df_exp = pd.read_csv(exp_path, sep='\t')

    # 调用函数，找到对应的基因的list
    gene_list = get_gene_list_by_pathway(workdir, species_name, pathway_name)

    # 根据基因的list,找到对应的表达谱,标准化为log2(n+1),保留两位小数
    df = df_exp[df_exp['Gene id'].isin(gene_list)]
    df.iloc[:, 1:] = df.iloc[:, 1:].applymap(lambda x: np.log2(x + 1)).round(2)

    return df


def get_gene_list_exp_fname_by_pathway(workdir: str, species_name: str, pathway_name: str) -> pd.DataFrame:
    """ 根据pathway的名字，依据exp_fname表，返回对应基因的表达谱
    """
    species_name = species_name.replace(' ', '_').lower()

    exp_fname_path = os.path.join(workdir, species_name, 'exp_fname.tsv')
    df_exp_fname = pd.read_csv(exp_fname_path, sep='\t')

    gene_list = get_gene_list_by_pathway(workdir, species_name, pathway_name)

    df = df_exp_fname[df_exp_fname['Gene id'].isin(gene_list)]
    df.iloc[:, 1:] = df.iloc[:, 1:].applymap(lambda x: np.log2(x + 1)).round(2)

    return df


def get_dataset_info_by_species_name(workdir: str, species_name: str) -> pd.DataFrame:
    """根据菌种的名字，返回dataset表
    """
    species_name = species_name.replace(' ', '_').lower()
    df_dataset = pd.read_csv(os.path.join(workdir, species_name, 'dataset.tsv'), sep='\t')
    
    return df_dataset


def get_sample_info_by_species_name(workdir: str, species_name: str) -> pd.DataFrame:
    """根据菌种的名字，返回sample表
    """
    species_name = species_name.replace(' ', '_').lower()
    df_sample = pd.read_csv(os.path.join(workdir, species_name, 'sample.tsv'), sep='\t')
    
    return df_sample

In [18]:
def get_pathway_id_by_pathway(workdir: str, species_name: str, pathway_name: str) -> str:
    """根据pathway的名字，依据kegg_id表，返回在相应pathway的id，生成pathway的通路图，保存在kegg-viewers目录下

    Args:
    * workdir: 工作目录
    * species_name: 菌种的名字
    * pathway_name: pathway的名字
    ----------
    Returns:
        pathway_id: pathway的id

    Usage: 
    安装：pip install kegg_viewer
    :   生成pathway的通路图的命令为：kegg_viewer -p pathway_id -O output -c cache
    :   pathway_id为pathway在kegg数据库中的id，output_dir为输出的通路图的目录，cache为缓存的目录
    :   通路图的svg矢量图输出目录为：workdir/species_name/kegg-viewers
    :   缓存的目录为：workdir/species_name/cache
    """
    # 空格换成_，大写转小写
    species_name = species_name.replace(' ', '_').lower()

    # 读取kegg_id表
    kegg_path = os.path.join(workdir, species_name,'data_processing_file/KEGG.csv')
    df_kegg_id = pd.read_csv(kegg_path, sep=',', names=['Id', 'Pathway'] )

    df = df_kegg_id[df_kegg_id['Pathway'] == pathway_name]
    pathway_id = df['Id'].tolist()[0]

    # try:
    #     os.system('kegg_viewer -p ' + pathway_id + 
    #               ' -O ' + os.path.join(workdir, species_name, 'kegg_viewers') + 
    #               ' -c ' + os.path.join(workdir, species_name, 'cache'))

    # except Exception as e:
    #     raise Exception("Failed to run kegg_viewer command. Error message: " + str(e))
    
    # 生成pathway的通路图
    try:
        output_dir = os.path.join(workdir, species_name, 'kegg_viewers')
        cache_dir = os.path.join(workdir, species_name, 'cache')
        
        cmd = f"kegg_viewer -p {pathway_id} -O {output_dir} -c {cache_dir}"
        subprocess.run(cmd, check=True, shell=True)

    except subprocess.CalledProcessError as e:
        raise Exception(f"Failed to run kegg_viewer command. Error message: {e}")

    return pathway_id

In [19]:
# 工作路径
workdir = '~/mtd/search'

# 假设前端传来的菌种名与通路名
species_name = 'Myceliophthora thermophila'

# 调用函数
# get_gene_list_by_pathway(workdir,species_name,pathway_name)
# get_gene_list_exp_by_pathway(workdir,species_name,pathway_name)
# get_gene_list_exp_fname_by_pathway(workdir,species_name,pathway_name)
# get_dataset_info_by_species_name(workdir,species_name)
# get_sample_info_by_species_name(workdir,species_name)